# Estimativas de Preços MOBIE


## Parametrização

In [1]:
import pandas as pd
from src.simulation.mobie.tariffs import load_tariffs_opc
from src.simulation.mobie.geo_functions import geoframe_distritos, dataframe_municipios
from src.simulation.mobie.vehicle_specs import vehicles
from src.simulation.mobie.prices import SimulatePricesMobie


EGME_OPC = 0.2499  # €/sessão
EGME_CEME = 0.2499  # €/sessão
IEC = 0.001 # €/kWh
VAT = 0.23

In [2]:
vehicle = vehicles.get("teslamodely")

NOM_VOLTAGE = vehicle["nominal_voltage"]
AVG_POWR = vehicle["avg_pwr_10_80"]
MAX_CAP = vehicle["max_cap"]
AC3_AMPS = vehicle["ac3_amps"]
AC_AMPS = vehicle["ac_amps"]


AVG_CURR = AVG_POWR * 1000 / NOM_VOLTAGE 
CAP_CHR = 0.7 * MAX_CAP
AVG_TIME = CAP_CHR / AVG_POWR * 60
TIME_AC3 = CAP_CHR / (AC3_AMPS * 230 * 3 / 1000 / 60)
TIME_AC = CAP_CHR / (AC_AMPS * 230 / 1000 / 60)
print(f"Energy: {CAP_CHR} kWh")
print(f"Average current (DC): {AVG_CURR:.1f}A")
print(f"Average charge time (DC): {AVG_TIME:.1f}min")
print(f"Average charge time (AC3): {TIME_AC3:.1f}min")
print(f"Average charge time (AC1): {TIME_AC:.1f}min")

Energy: 42.0 kWh
Average current (DC): 294.1A
Average charge time (DC): 25.2min
Average charge time (AC3): 228.3min
Average charge time (AC1): 342.4min


## Extração e Preparação de Dados

### Geografia

In [3]:
gdf_distritos = geoframe_distritos()
df_municipios = dataframe_municipios()

reading from geoapi
reading from geoapi



### Tarifas MobiE

In [4]:
tariffs, meta = load_tariffs_opc("data/naps/portugal/simulation/tarifas.csv")

tariffs.head(5).T

def merge_municipios(left, right, how="left"):
    left["_merge_col"] = left["MUNICIPIO"].str.normalize("NFKD").str.encode('ascii', errors='ignore').str.decode('utf-8').str.capitalize()
    right["_merge_col"] = right["MUNICIPIO"].str.normalize("NFKD").str.encode('ascii', errors='ignore').str.decode('utf-8').str.capitalize()
    right = right.rename(columns={"MUNICIPIO": "MUNICIPIO_GEOJS"})
    left = left.merge(right, on=["_merge_col"], how=how)
    return left.drop(columns=["_merge_col"])
    

meta = merge_municipios(meta, df_municipios)
meta

,TIPO_POSTO,MUNICIPIO,MORADA,OPERADOR,MOBICHARGER,NIVELTENSAO,TIPO_TOMADA,FORMATO_TOMADA,POTENCIA_TOMADA,id,idx_evse,idx_connector,DISTRITO,MUNICIPIO_GEOJS
0,Semirrápido,Albufeira,Vila Galé Cerro Alagoa,EDP,False,MT,MENNEKES,SOCKET,"7,4",ABF-00008,1,1,Faro,Albufeira
1,Semirrápido,Albufeira,Vila Galé Cerroa Alagoa,EDP,False,MT,MENNEKES,SOCKET,"7,4",ABF-00009,1,1,Faro,Albufeira
2,Semirrápido,Albufeira,Praia Maria Luísa,BLU,False,BTE,MENNEKES,SOCKET,22,ABF-00011,1,1,Faro,Albufeira
3,Semirrápido,Albufeira,Praia Maria Luísa,BLU,False,BTE,MENNEKES,SOCKET,22,ABF-00011,2,1,Faro,Albufeira
4,Semirrápido,Albufeira,Makro Albufeira,GLP,False,MT,MENNEKES,SOCKET,22,ABF-00012,1,1,Faro,Albufeira
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10927,Rápido,Vouzela,"A25 / IP5, KM 7.9",GLG,False,BTE,CHADEMO,CABLE,50,VZL-00002,1,1,Viseu,Vouzela
10928,Rápido,Vouzela,"A25 / IP5, KM 7.9",GLG,False,BTE,CCS,CABLE,50,VZL-00002,2,1,Viseu,Vouzela
10929,Rápido,Vouzela,"A25 / IP5, KM 7.9",GLG,False,BTE,MENNEKES,CABLE,43,VZL-00002,3,1,Viseu,Vouzela
10930,Semirrápido,Vouzela,R. Bombeiros Voluntários,ATL,False,BTN,MENNEKES,SOCKET,22,VZL-80003,1,1,Viseu,Vouzela


In [5]:
opcs = pd.read_csv("data/naps/portugal/simulation/opcs.csv", sep=";")[["party_id", "party_name"]]
opcs

,party_id,party_name
0,ALF,Alfa Energia
1,ATL,Atlante
2,BPP,BP
3,BLU,Blue Charge
4,INT,Bluint
...,...,...
99,PTE,Ynerluz
100,ZUN,Zunder
101,PAC,"PACORP, LDA"
102,FRI,"Distrifrio Supermercados, Lda"


## Simulação de Preços

In [6]:
simulator = SimulatePricesMobie(
    path_connectors="data/naps/portugal/locations.json",
    path_tariffs="data/naps/portugal/simulation/tarifas.csv",
    fee_egme_ceme=EGME_CEME,
    fee_egme_opc=EGME_OPC,
    iec=IEC,
    vat=VAT,
)

simulator.load_connectors()
simulator.load_opc_tariffs()
simulator.load_tariffs_ceme()
simulator.set_vehicle(vehicle=vehicle)
simulator.add_vehicle_params()

simulator.estimate_opc_cost()
simulator.estimate_ceme_cost(w_vazio=0)
(
    simulator.estimate_opc
).head(2).T

,0,1
id,ABF-00008,ABF-00009
party_id,EDP,EDP
idx_evse,1,1
idx_connector,1,1
mobie_voltage_level,MT,MT
evse_id,PT*EDP*E*ABF*00008*1,PT*EDP*E*ABF*00009*1
standard,IEC_62196_T2,IEC_62196_T2
max_voltage,240,240
max_amperage,32,32
max_electric_power,7.4,7.4
